# Tái lập GraphVis — Kaggle / Colab (GPU miễn phí)

Chạy **từ trên xuống**. Mỗi ô thực nghiệm có thể chạy lại sau khi phiên bị ngắt: dữ liệu đã dựng và
giai đoạn đã huấn luyện xong được bỏ qua, giai đoạn dở dang resume từ checkpoint.

| Ô | Thực nghiệm | Mục báo cáo |
|---|---|---|
| G0 | dựng dữ liệu (đồ thị con, ảnh, D_g, D_v) | VII-A |
| G1a | mô hình gốc: text / image / KAPING trên CSQA, OBQA | VII-B |
| G1b, G1c | GraphVis curriculum trên CSQA, OBQA | VII-B |
| G6 | 6 tác vụ đọc ảnh trước/sau giai đoạn 1 | VII-E |
| G2 | ScienceQA-IMG, MMBench, POPE | VII-C |
| G4 | joint / graph-first / OCR-first | VII-D |
| G3 | ScienceQA cho bản joint | VII-C |
| G5 | kết hợp KAPING | VII-D |
| G7 | ảnh 50×50 | VII-E |
| C3 | đối chứng tinh chỉnh không ảnh | VI-C |
| C1 | độ chính xác theo số nút | VI-C |
| C2 | độ chính xác theo độ phân giải | VI-C |

**Trước khi chạy:** Kaggle → *Settings*: bật Internet, chọn GPU (T4 x2 hoặc P100); thêm *Secret* `WANDB_API_KEY`
(và `HF_TOKEN` nếu dùng `HUB_REPO`). Colab → *Runtime type* GPU; thêm secret cùng tên trong mục 🔑.

In [ ]:
# ==== CẤU HÌNH ====
SCALE = "lite"          # tiny (thử nhanh) | lite (mặc định, ~5-7 giờ/lượt train trên T4) | full (như bài)
QUANT, DTYPE = "4bit", "fp16"   # A100/L4: "none", "bf16"
WANDB_PROJECT = "graphml-seminar-report"
WANDB_ENTITY = ""       # để trống = tài khoản mặc định
HUB_REPO = ""           # vd "your-hf-name/graphvis-adapters" để lưu adapter lên HF Hub (chuyển giữa Kaggle/Colab)
REPO_URL = "https://github.com/AuDuongKhang/graphml-seminar-report.git"
BRANCH = "main"
VQA_MAX = {"tiny": {"scienceqa": 50, "mmbench": 20, "pope": 50},
           "lite": {"scienceqa": None, "mmbench": 400, "pope": 500},
           "full": {"scienceqa": None, "mmbench": None, "pope": None}}[SCALE]

In [ ]:
import os, sys, subprocess
ON_KAGGLE = os.path.exists("/kaggle/working")
ON_COLAB = "google.colab" in sys.modules
if ON_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE = "/content/drive/MyDrive/graphml-repro"
elif ON_KAGGLE:
    BASE = "/kaggle/working/graphml-repro"
else:
    BASE = os.path.abspath("graphml-repro")
os.makedirs(BASE, exist_ok=True)
os.environ.update(GRAPHML_DATA=f"{BASE}/data", GRAPHML_OUT=f"{BASE}/out", WANDB_PROJECT=WANDB_PROJECT)
if WANDB_ENTITY:
    os.environ["WANDB_ENTITY"] = WANDB_ENTITY
REPO = f"{BASE}/repo"
if not os.path.exists(REPO):
    subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL, REPO], check=True)
else:
    subprocess.run(["git", "-C", REPO, "pull"], check=True)
GV = f"{REPO}/repro/graphvis"
OUT = os.environ["GRAPHML_OUT"] + "/graphvis"
print("BASE =", BASE)

In [ ]:
!apt-get -qq install -y graphviz > /dev/null
!pip install -q -r {GV}/requirements.txt

In [ ]:
def secret(name):
    try:
        if ON_KAGGLE:
            from kaggle_secrets import UserSecretsClient
            return UserSecretsClient().get_secret(name)
        if ON_COLAB:
            from google.colab import userdata
            return userdata.get(name)
    except Exception:
        return os.environ.get(name)
    return os.environ.get(name)

os.environ["WANDB_API_KEY"] = secret("WANDB_API_KEY") or ""
if HUB_REPO:
    os.environ["HF_TOKEN"] = secret("HF_TOKEN") or ""
import wandb; wandb.login()
M = f"--quant {QUANT} --dtype {DTYPE}"
S = f"--set quant={QUANT} dtype={DTYPE}" + (f" hub_repo={HUB_REPO}" if HUB_REPO else "")
def adapter(exp, stage):
    return f"{OUT}/{exp}/{stage}/final"
%cd {GV}

## Kiểm tra nhanh (chạy 1 lần): test CPU + test GPU + dry-run huấn luyện

In [ ]:
!cd {REPO}/repro && python -m pytest -q
!cd {REPO}/repro && python -m pytest -q -m gpu graphvis/tests/test_collate.py -p no:cacheprovider
!python -m gv.cli build-data --dataset csqa --split train --scale tiny --tag smoke --no_wandb
!python -m gv.cli train --config configs/G1_csqa_curriculum.yaml --dry_run {S} exp_id=SMOKE data_tag=smoke

## G0 — Dựng dữ liệu

In [ ]:
for ds, sp in [("csqa", "train"), ("csqa", "dev"), ("obqa", "train"), ("obqa", "test")]:
    !python -m gv.cli build-data --dataset {ds} --split {sp} --scale {SCALE}

## G1a — Mô hình gốc (không tinh chỉnh): text / image / KAPING

In [ ]:
for ds, sp in [("csqa", "dev"), ("obqa", "test")]:
    for mode in ["text", "image", "kaping"]:
        !python -m gv.cli eval-mcqa --dataset {ds} --split {sp} --mode {mode} --exp_id G1a-base-{ds}-{mode} --group G1 {M}

## G1b — GraphVis curriculum trên CSQA (mô hình chính)

In [ ]:
!python -m gv.cli train --config configs/G1_csqa_curriculum.yaml {S}
!python -m gv.cli eval-mcqa --dataset csqa --split dev --mode image --adapter {adapter('G1-csqa-curriculum','stage2_qa')} --exp_id G1b-graphvis-csqa-image --group G1 {M}

## G1c — GraphVis curriculum trên OpenBookQA

In [ ]:
!python -m gv.cli train --config configs/G1_obqa_curriculum.yaml {S}
!python -m gv.cli eval-mcqa --dataset obqa --split test --mode image --adapter {adapter('G1-obqa-curriculum','stage2_qa')} --exp_id G1c-graphvis-obqa-image --group G1 {M}

## G6 — Đọc ảnh đồ thị: mô hình gốc vs sau giai đoạn 1

In [ ]:
!python -m gv.cli eval-graph-reading --exp_id G6-base --group G6 --batch_size 4 {M}
!python -m gv.cli eval-graph-reading --adapter {adapter('G1-csqa-curriculum','stage1_graph_reading')} --exp_id G6-after-stage1 --group G6 --batch_size 4 {M}

## G2 — VQA zero-shot: mô hình gốc vs GraphVis (CSQA)

In [ ]:
for name, ad in [("base", "none"), ("graphvis", adapter('G1-csqa-curriculum','stage2_qa'))]:
    for b in ["scienceqa", "mmbench", "pope"]:
        mx = f"--max_eval {VQA_MAX[b]}" if VQA_MAX[b] else ""
        !python -m gv.cli eval-vqa --benchmark {b} --adapter {ad} --exp_id G2-{name}-{b} --group G2 {mx} {M}

## G4 — Ablation thứ tự huấn luyện (Fig 7)

In [ ]:
for cfg, exp, stage in [("G4_joint", "G4-csqa-joint", "joint"),
                        ("G4_graph_first", "G4-csqa-graph-first", "stage2_qa"),
                        ("G4_ocr_first", "G4-csqa-ocr-first", "stage2_qa")]:
    !python -m gv.cli train --config configs/{cfg}.yaml {S}
    !python -m gv.cli eval-mcqa --dataset csqa --split dev --mode image --adapter {adapter(exp, stage)} --exp_id {exp}-eval --group G4 {M}

## G3 — ScienceQA cho bản joint (Table 6)

In [ ]:
!python -m gv.cli eval-vqa --benchmark scienceqa --adapter {adapter('G4-csqa-joint','joint')} --exp_id G3-joint-scienceqa --group G3 {M}

## G5 — Kết hợp KAPING (Table 3)

In [ ]:
for name, ad in [("base", "none"), ("joint", adapter('G4-csqa-joint','joint')),
                 ("curriculum", adapter('G1-csqa-curriculum','stage2_qa'))]:
    !python -m gv.cli eval-mcqa --dataset csqa --split dev --mode image+kaping --adapter {ad} --exp_id G5-{name}-kaping --group G5 {M}

## G7 — Ảnh đồ thị 50×50 (Fig 9)

In [ ]:
for sp in ["train", "dev"]:
    !python -m gv.cli build-data --dataset csqa --split {sp} --scale {SCALE} --image_size 50 --tag lowres50
!python -m gv.cli train --config configs/G7_lowres.yaml {S}
!python -m gv.cli eval-mcqa --dataset csqa --split dev --mode image --data_tag lowres50 --adapter {adapter('G7-csqa-lowres50','stage2_qa')} --exp_id G7-lowres50-eval --group G7 {M}

## C3 — Đối chứng: tinh chỉnh không ảnh

In [ ]:
!python -m gv.cli train --config configs/C3_text_only.yaml {S}
!python -m gv.cli eval-mcqa --dataset csqa --split dev --mode text --adapter {adapter('C3-csqa-text-only','text_qa')} --exp_id C3-text-only-eval --group C3 {M}

## C1 — Độ chính xác theo số nút (dùng lại dự đoán đã lưu)

In [ ]:
!python -m gv.cli c1-report --eval_exp_ids G1a-base-csqa-text G1a-base-csqa-image G1b-graphvis-csqa-image C3-text-only-eval G7-lowres50-eval

## C2 — Độ chính xác theo độ phân giải ảnh

In [ ]:
!python -m gv.cli resolution-sweep --exp_prefix C2-base {M}
!python -m gv.cli resolution-sweep --adapter {adapter('G1-csqa-curriculum','stage2_qa')} --exp_prefix C2-graphvis {M}